Adding Anchor(pos_emb at recursion)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import copy

# ==========================================
# 1. Diffusion Utilities (Sqrt Schedule)
# ==========================================
def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """
    Sqrt noise schedule from Diffusion-LM (Appendix A).
    alpha_bar_t = 1 - sqrt(t/T + s)
    """
    # t is expected to be a tensor of shape (batch_size, 1, 1)
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    # Clamp to prevent negative variances or exact zeros
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

class EMA:
    """Exponential Moving Average for model weights (Crucial for TRM stability)"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================================
# 2. Model Architecture (TRM + Diffusion)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        x = self.norm1(x + self.out_proj(attn_out))
        x = self.norm2(x + self.mlp(x))
        return x

class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # End-to-End Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # FREEZE EMBEDDINGS to force the TRM layers to learn denoising.
        # Once your loss curve looks normal and outputs are coherent, set this to True.
        self.token_emb.weight.requires_grad = True
        self.pos_emb = nn.Embedding(1024, d_model) 
        
        # Timestep embedding (Diffusion specific)
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # TRM Backbone (Less is More - keeping layers relatively low)
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        # Continuous Output Head (Predicting x_0)
        self.output_head = nn.Linear(d_model, d_model)
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))

        # ==========================================
        # NEW: DISCRETE OUTPUT HEAD (Predicting words)
        # ==========================================
        # This maps the continuous prediction back to vocabulary logits
        self.lm_head = nn.Linear(d_model, vocab_size)


    def get_sinusoidal_embeddings(self, t):
        """Standard sinusoidal positional embeddings for time"""
        half_dim = self.d_model // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def latent_recursion(self, x_t_emb, z, t_emb, n, pos_embeddings):
        """Inner reasoning loop with the Positional Syntax Anchor"""
        for _ in range(n):
            # THE ANCHOR: Add pos_embeddings at every single pass
            combined_state = x_t_emb + z + t_emb.unsqueeze(1) + pos_embeddings
            
            for layer in self.layers:
                combined_state = layer(combined_state)
            z = combined_state
            
        pred_x0_emb = self.output_head(z)
        return pred_x0_emb, z

    def deep_recursion(self, x_t_emb, z, t_emb, n, T, pos_embeddings):
        """Outer cycle that passes pos_embeddings down"""
        for _ in range(T):
            pred_x0_emb, z = self.latent_recursion(x_t_emb, z, t_emb, n, pos_embeddings)
        return pred_x0_emb, z

    def forward(self, x_t_emb, t, positions, n=6, T=3): # Note the new defaults!
        t_emb = self.time_mlp(t)
        pos_embeddings = self.pos_emb(positions)
        
        # Initialize z (usually zeros)
        z = torch.zeros_like(x_t_emb)
        
        # Run the deep recursion WITH the positional embeddings
        pred_x0_emb, z_final = self.deep_recursion(x_t_emb, z, t_emb, n, T, pos_embeddings)
        
        return pred_x0_emb

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast

class TinyStoriesLocalDataset(Dataset):
    def __init__(self, file_path, tokenizer, seq_len=64, limit=1000000):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.stories = []
        
        print(f"Loading up to {limit} stories from {file_path}...")
        with open(file_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= limit:
                    break
                
                line = line.strip()
                if line: # Skip empty lines
                    self.stories.append(line)
                    
        print(f"Successfully loaded {len(self.stories)} stories into RAM.")

    def __len__(self):
        return len(self.stories)

    def __getitem__(self, idx):
        text = self.stories[idx]
        
        # Tokenize on the fly. PreTrainedTokenizerFast is heavily optimized in Rust,
        # so doing this per-item during training is extremely efficient.
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        # Squeeze to remove the batch dimension added by the tokenizer
        return {"input_ids": encoding["input_ids"].squeeze(0)}

def get_tinystories_dataloader(file_path="tinystories_dataset.txt", batch_size=32, seq_len=256):
    """Loads local TinyStories text file and tokenizes with custom Word-Level tokenizer"""
    
    print("Initializing Custom Word-Level Tokenizer...")
    tokenizer = PreTrainedTokenizerFast(tokenizer_file="tinystories_wordlevel.json")
    
    # Explicitly map the special control tokens
    tokenizer.add_special_tokens({
        'unk_token': '[UNK]',
        'pad_token': '[PAD]',
        'bos_token': '[BOS]',
        'eos_token': '[EOS]'
    })
    
    vocab_size = len(tokenizer)
    
    # Initialize the custom dataset
    dataset = TinyStoriesLocalDataset(
        file_path=file_path, 
        tokenizer=tokenizer, 
        seq_len=seq_len, 
        limit=1000000
    )
    
    # Create the PyTorch DataLoader
    # Note: num_workers=2 or 4 can speed up data loading if your CPU allows it
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    
    return dataloader, vocab_size

In [ ]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm

# ==========================================
# 4. Training Loop (Experiment B)
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    
    # --- MEMORY SETTINGS ---
    batch_size = 32           # Micro-batch size to fit in VRAM
    accumulation_steps = 4    # 32 * 4 = 128 Effective Batch Size
    
    T_DIFFUSION = 2000
    EPOCHS = 1 
    
    # --- DEEP RECURSION SETTINGS ---
    N_RECURSIONS = 6
    T_CYCLES = 3
    
    # 1. Load Custom TinyStories DataLoader
    # Make sure get_tinystories_dataloader is imported/defined above this!
    dataloader, vocab_size = get_tinystories_dataloader(
        file_path="tinystories_dataset.txt", 
        batch_size=batch_size, 
        seq_len=seq_len
    )
    
    # 2. Initialize the Model
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    
    # 3. FREEZE EMBEDDINGS (Training Wheels ON for Epoch 0)
    model.token_emb.weight.requires_grad = False
    
    # 4. Setup Optimizer (Only pass parameters that require gradients)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="Experiment-B-Deep-Recursion",
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "effective_batch_size": batch_size * accumulation_steps,
            "micro_batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 1e-4,
            "N_recursions": N_RECURSIONS,
            "T_cycles": T_CYCLES
        }
    )
    
    print("Starting Experiment B: Deep Recursion with Syntax Anchor...")
    model.train()
    global_step = 0 
    
    # Zero out gradients before the loop starts
    optimizer.zero_grad()
    
    for epoch in range(EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings & Positional Anchor
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            
            # Note: We still add pos_emb here as the baseline, but the model 
            # will also re-inject them internally during the latent_recursion loop.
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # ==========================================
            # 3. DEEP RECURSION FORWARD PASS
            # ==========================================
            # This now routes through your standard forward() method 
            # and passes positions down to the syntax anchor
            pred_x0_emb, _ = model(x_t_emb, t, positions, n=N_RECURSIONS, T=T_CYCLES)

            # Project to Discrete Vocabulary
            logits = model.lm_head(pred_x0_emb) 
            
            # 4. Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_ce = F.cross_entropy(logits.view(-1, vocab_size), input_ids.view(-1))
            total_loss = loss_recon + (0.5 * loss_ce)
            
            # ==========================================
            # 5. GRADIENT ACCUMULATION BACKPROPAGATION
            # ==========================================
            # Scale the loss down because we are summing gradients over multiple micro-batches
            loss = total_loss / accumulation_steps
            loss.backward()
            
            # Only step the optimizer and update EMA after 'accumulation_steps' micro-batches
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                ema.update() # Update EMA only when weights actually change
                
                # Update metrics live
                wandb.log({
                    "train/total_loss": total_loss.item(),
                    "train/loss_recon": loss_recon.item(),
                    "train/loss_ce_discrete": loss_ce.item(),
                    "epoch": epoch,
                    "global_step": global_step
                })
                
                global_step += 1
                
                # Mid-epoch save (Checked against global_step to align with actual weight updates)
                if global_step > 0 and global_step % 5000 == 0:
                    ema.apply_shadow()
                    save_path = f"trm_diffusion_step_{global_step}.pt"
                    torch.save({
                        'epoch': epoch,
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                    }, save_path)
                    wandb.save(save_path) 
                    ema.restore()
            
            # Update the right side of the progress bar with the unscaled total loss
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})

        # End of Epoch Save
        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

In [ ]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm

# ==========================================
# Phase 2: Semantic Fine-Tuning (Epoch 2-10)
# ==========================================
def train_phase_2():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    
    # --- MEMORY SETTINGS ---
    batch_size = 32           
    accumulation_steps = 4    
    
    T_DIFFUSION = 2000
    TOTAL_EPOCHS = 10 
    START_EPOCH = 1 # 0-indexed, so 1 means we are starting at Epoch 2
    
    # --- DEEP RECURSION SETTINGS ---
    N_RECURSIONS = 6
    T_CYCLES = 3
    
    # 1. Load Custom TinyStories DataLoader
    dataloader, vocab_size = get_tinystories_dataloader(
        file_path="tinystories_dataset.txt", 
        batch_size=batch_size, 
        seq_len=seq_len
    )
    
    # 2. Initialize the Model
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    
    # 3. LOAD SAVED WEIGHTS FROM EPOCH 1
    checkpoint_path = "trm_diffusion_epoch_1_complete.pt"
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Safely load state dict, handling both nested dictionary and direct state_dict formats
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    print("Model weights successfully loaded!")
    
    # 4. THE CRITICAL FLIP: UNFREEZE THE EMBEDDINGS
    model.token_emb.weight.requires_grad = True
    print("Embeddings unfrozen! Model will now learn semantic grammar clusters.")
    
    # 5. Initialize a NEW Optimizer with a lower learning rate
    # We must do this because AdamW needs to track the newly unfrozen embeddings
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="Experiment-B-Phase-2-Unfrozen",
        config={
            "dataset_size": "1M",
            "epochs": TOTAL_EPOCHS,
            "starting_epoch": START_EPOCH + 1,
            "effective_batch_size": batch_size * accumulation_steps,
            "micro_batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 5e-5, # Log the dropped LR
            "N_recursions": N_RECURSIONS,
            "T_cycles": T_CYCLES,
            "embeddings_frozen": False
        }
    )
    
    print("Starting Phase 2 Training...")
    model.train()
    
    # Pick up the global step where Epoch 1 left off (roughly ~7800 steps for 1M stories at BS=128)
    global_step = checkpoint.get('global_step', 0)
    optimizer.zero_grad()
    
    for epoch in range(START_EPOCH, TOTAL_EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # Clean x_0 Embeddings & Positional Anchor
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # DEEP RECURSION FORWARD PASS
            pred_x0_emb, _ = model(x_t_emb, t, positions, n=N_RECURSIONS, T=T_CYCLES)

            # Project to Discrete Vocabulary
            logits = model.lm_head(pred_x0_emb) 
            
            # Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_ce = F.cross_entropy(logits.view(-1, vocab_size), input_ids.view(-1))
            total_loss = loss_recon + (0.5 * loss_ce)
            
            # GRADIENT ACCUMULATION BACKPROPAGATION
            loss = total_loss / accumulation_steps
            loss.backward()
            
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                ema.update() 
                
                wandb.log({
                    "train/total_loss": total_loss.item(),
                    "train/loss_recon": loss_recon.item(),
                    "train/loss_ce_discrete": loss_ce.item(),
                    "epoch": epoch,
                    "global_step": global_step
                })
                
                global_step += 1
                
                if global_step > 0 and global_step % 5000 == 0:
                    ema.apply_shadow()
                    save_path = f"trm_diffusion_phase2_step_{global_step}.pt"
                    torch.save({
                        'epoch': epoch,
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                    }, save_path)
                    wandb.save(save_path) 
                    ema.restore()
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train_phase_2()